# Permutation Tests

In this notebook we will demonstrate how to use acore's permutation testing functions on
metagenomics data collected by [Ju and colleagues
(2018)](https://doi.org/10.1038/s41396-018-0277-8).

The samples in this demo were collected from wastewater treatment plant influent
(MGYS00005056) and effluent (MGYS00005058).

For this demo we look at the GO term abundance tables generated by the Mgnify pipeline.
The values in the table are the absolute abundance of selected GO terms for each sample,
which we then transform to relative abundances and centred-log ratios.

## Data preparation details

### Downloading
The analysed samples were downloaded via the [MGnify
API](https://www.ebi.ac.uk/metagenomics/api/docs/). The inffluent (INF) and effluent
(EFF) datasets have paired samples and we also needed to download the sample metadata
(also available via Mgnify API) to assign the correct pairing.

### Preprocessing of abundances
- To account for technical variation due to sequencing technology limitations, we first
  transform the abundance values so they are relative to the total reads for the sample
  aka getting relative abundances.
- The relative abundances are compositional data (CoDa) so we map them to unconstrained
  vectors using centred log-ratio transformation
  [`acore.transform.compositional.calc_clr`](acore.transform.compositional)
  to not violate assumptions of any frequentist stats we do

### Preprocessing of the metadata
- the sample metadata needed for this demo (sampling location) were available in their
  "sample-desc"
- the sample-desc for each sample in both INF and EFF were parsed and used for pairing
  off

## Subset of data for demo
- For this demo we only look at [go term
  GO:0017001](https://www.ebi.ac.uk/QuickGO/term/GO:0017001)
- It's expected that antibiotic catabolic processes to be higher in influent (INF) vs
  effluent (EFF) samples.

### Saved the demo dataset
This example subset of data was saved to a CSV,
[`Ju2018_GO0017001_enf_inf_paired.csv`](example_data/mgnify/Ju2018_GO0017001_enf_inf_paired.csv).
The data dictionary is below:

| column            | description                                                                                                       | dtype |
|-------------------|-------------------------------------------------------------------------------------------------------------------|-------|
| eff_id            | The run id for the mgnify analysis of the effluent sample.                                                        | str   |
| inf_id            | The run id for the mgnify analysis of the influent sample.                                                        | str   |
| sampling_location | [The ISO 3166-1 alpha-2 code](http://iso.org/obp/ui/#iso:pub:PUB500001:en) for the country where the sample was from. | str   |
| sampling_read     | Replicates?                                                                                                       | str   |
| eff_abundance     | The relative abundance of the GO term for a given effluent sample following preprocessing (i.e., CoDA and CLR)    | float |
| inf_abundance     | The relative abundance of the GO term for a given influent sample following preprocessing (i.e., CoDA and CLR)    | float |

-----

We will now proceed with reading in the prepared dataset.

In [1]:
from pprint import pprint

import numpy as np
import pandas as pd

from acore.permutation_test import (
    chi2_permutation,
    indep_permutation,
    paired_permutation,
)

df_data = pd.read_csv(
    "https://raw.githubusercontent.com/Multiomics-Analytics-Group/acore/refs/heads/anglup-learning/"
    "example_data/mgnify/Ju2018_GO0017001_enf_inf_paired.csv"
)
# sanity check
df_data

,eff_id,inf_id,sampling_location,sampling_read,eff_abundance,inf_abundance
0,ERR2985255,ERR2814663,TG,READ2 Taxonomy ID:256318,3.257283,4.226819
1,ERR2985256,ERR2814664,MN,READ2 Taxonomy ID:256318,2.572841,3.847191
2,ERR2985257,ERR2814651,AH,READ1 Taxonomy ID:256318,4.298777,4.086841
3,ERR2985258,ERR2814667,TE,READ1 Taxonomy ID:256318,2.758982,3.436752
4,ERR2985259,ERR2814660,FD,READ1 Taxonomy ID:256318,3.364675,3.486673


## Paired permutation test

Since these are paired samples we will proceed with paired sample permutation test using
[`acore.permutation_test.paired_permutation()`](acore.permutation_test.paired_permutation).

The permutation test compares the actual observed chosen metric (e.g., t-statistic, mean
difference) with metrics calculated when the dataset values are randomly shuffled
permutations of the dataset.

If we do 100 permutations of our data (although we should do a bunch more) and only 1 of
those permutations falsely showed a larger effect size than the actual observed effect
than it suggests there is a 1/100 chance (p value of 0.01) of the observed effect size
having occurred by chance.

Optional choice of random number generator for reproducibility.

In [2]:
rng = np.random.default_rng(12345)
for metric in ["t-statistic", "mean", np.mean]:
    result = paired_permutation(
        df_data["inf_abundance"],
        df_data["eff_abundance"],
        metric=metric,
        n_permutations=1000,
        rng=rng,
    )
    # verbosity
    pprint(result)

Based on the permutation tests by test statistic and mean difference, the probability of
the observed metrics (t=6.739 and mean diff=0.535) occurring at random would be
<0.00001.

## Independent sample permutation test

`paired_permutation` only makes sense when observations have a one-to-one
correspondence, as our influent/effluent samples do. When two groups are unrelated
(no pairing), we should use
[`acore.permutation_test.indep_permutation()`](acore.permutation_test.indep_permutation)
instead.

Our samples were also each sequenced in one of two runs (`sampling_read`: READ1 vs
READ2). Those two read groups can be considere not paired with one another for
illustration purposes, so we use them to demonstrate the independent-sample test: is
effluent GO term abundance affected by which read the sample came from?

`indep_permutation` requires its inputs as numpy arrays.

In [ ]:
read1_mask = df_data["sampling_read"] == "READ1 Taxonomy ID:256318"
read2_mask = df_data["sampling_read"] == "READ2 Taxonomy ID:256318"

read1_abundance = df_data.loc[read1_mask, "eff_abundance"]
read2_abundance = df_data.loc[read2_mask, "eff_abundance"]

rng = np.random.default_rng(12345)
for metric in ["t-statistic", "anova", "mean", "median"]:
    result = indep_permutation(
        read1_abundance,
        read2_abundance,
        metric=metric,
        n_permutations=1000,
        rng=rng,
    )
    # verbosity
    pprint(result)

Across all four metrics the permutation p-values are well above 0.05, so — unlike the
influent vs effluent comparison — we find no evidence that the sequencing read group
affects the measured abundance - as expected for technical artefact.

## Chi-squared permutation test

[`acore.permutation_test.chi2_permutation()`](acore.permutation_test.chi2_permutation)
tests whether the distribution of *categorical* observations differs between groups,
using a permuted chi-squared statistic on a contingency table. Our abundance values are
continuous, so we first discretise them into "Low" vs "High" based on the median to
obtain something categorical to compare.

We reuse the same READ1 vs READ2 grouping as above: does the proportion of "Low"/"High"
effluent abundance samples differ between the two read groups?

In [ ]:
df_data["abundance_level"] = pd.cut(
    df_data["eff_abundance"],
    bins=[-np.inf, df_data["eff_abundance"].median(), np.inf],
    labels=["Low", "High"],
)
df_data

In [ ]:
read1_level = df_data.loc[read1_mask, "abundance_level"]
read2_level = df_data.loc[read2_mask, "abundance_level"]

result = chi2_permutation(
    read1_level, read2_level, n_permutations=1000, rng=np.random.default_rng(12345)
)
# verbosity
pprint(result)

The permutation p-value is well above 0.05, so, as with the independent-sample test
above, we find no evidence that the read group changes the proportion of "Low"/"High"
abundance samples.